# Python tutorial: execute the Rust scheduler

Python orchestrates the actual Rust engine through `NativeEngine`. Rust is the only supported simulation core. The former pure-Python `Scheduler` is unsupported immediately; this notebook has no simulation fallback or synthetic success path.

The diagram below illustrates event scheduling. It is not captured output from this notebook.

![Scheduler timeline](figures/scheduler-timeline.svg)


## Prepare the native library and notebook kernel

Follow [installation](../docs/install.md) and the [Python binding README](../bindings/python/README.md) to build the matching Rust C ABI library and install the binding into the notebook kernel's qualified Python environment. Set `KAIRO_ECS_FFI_LIBRARY` to the **absolute path** of the built shared library before running cells.

A missing library is an error. A path check is not native qualification: `NativeEngine` also loads the library and checks ABI version and required symbols. Retain the source revision, library checksum, build features and Python version when recording native acceptance.


In [ ]:
import os
from pathlib import Path

library_value = os.environ.get("KAIRO_ECS_FFI_LIBRARY")
if not library_value:
    raise RuntimeError(
        "This notebook requires the actual Rust engine. Build the matching C ABI "
        "library and set KAIRO_ECS_FFI_LIBRARY to its absolute path; no fallback is supported."
    )
native_library = Path(library_value)
if not native_library.is_absolute() or not native_library.is_file():
    raise RuntimeError("KAIRO_ECS_FFI_LIBRARY must name an existing absolute library path.")

from kairo_ecs import NativeEngine


## Schedule and advance in Rust

The Rust scheduler orders events by tick, priority and insertion sequence. The current Python native adapter exposes opaque event handles, dispatch counts and statistics. This example verifies bounded execution and time through that interface; it does not claim to observe a detailed event trace or prove equal-time priority ordering from counts alone.

The C ABI accepts unsigned 64-bit ticks, narrower than Rust's unsigned 128-bit `SimTime`. The context manager releases the native handle.


In [ ]:
with NativeEngine(lib_path=str(native_library)) as engine:
    handles = [
        engine.schedule_at(10, priority=2, kind=3),
        engine.schedule_at(5, priority=9, kind=1),
        engine.schedule_at(10, priority=1, kind=2),
        engine.schedule_at(10, priority=1, kind=4),
    ]
    assert len(set(handles)) == 4
    assert all(handle != 0 for handle in handles)
    assert engine.run_until(5) == 1
    assert engine.current_time_ticks == 5
    assert engine.stats()["pending_events"] == 3
    assert engine.run_for(2) == 2
    assert engine.stats()["pending_events"] == 1
    assert engine.run_for(1) == 1
    assert engine.step() is False
    scheduling_stats = engine.stats()
    assert scheduling_stats == {
        "current_time_ticks": 10,
        "scheduled_events": 4,
        "dispatched_events": 4,
        "cancelled_events": 0,
        "pending_events": 0,
    }

scheduling_stats


## Cancel Rust-owned pending work

Cancellation uses the opaque handle returned by Rust. A repeated cancellation or cancellation after dispatch returns `False`. No host-side queue, reconstructed event identifier or synthetic trace is used.


In [ ]:
with NativeEngine(lib_path=str(native_library)) as engine:
    first = engine.schedule_at(1, kind=10)
    cancelled = engine.schedule_at(2, kind=20)
    third = engine.schedule_at(3, kind=30)

    assert engine.cancel(cancelled) is True
    assert engine.cancel(cancelled) is False
    assert engine.run_for(10) == 2
    assert engine.cancel(first) is False
    assert engine.cancel(third) is False
    cancellation_stats = engine.stats()
    assert cancellation_stats == {
        "current_time_ticks": 3,
        "scheduled_events": 3,
        "dispatched_events": 2,
        "cancelled_events": 1,
        "pending_events": 0,
    }

cancellation_stats


## Interpret the result at its actual scope

If the executable cells pass, they exercised the loaded Rust library's scheduling, bounded execution, cancellation, statistics and handle lifetime for these inputs. They did not establish arbitrary model callbacks, trace export, cross-platform determinism, inference, reinforcement learning or performance superiority.

`notebooks/validate_notebooks.py` parses notebook structure and Python syntax without executing these cells. A passing metadata check is not native runtime evidence. No execution outputs are committed here; record real acceptance output only after running with the built artefact.

Continue with [Rust scheduling](../docs/tutorials/rust-getting-started.md), [current capabilities](../docs/current-state.md) and [model-building guidance](../docs/tutorials/model-building.md).
